In [13]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict

In [14]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [28]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluate: str

In [29]:
def create_outline(state: BlogState) -> BlogState:
    
    # Fetch title
    title = state['title']

    # Call llm gen outline
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    chain = llm | StrOutputParser()

    # Update state
    state['outline'] = chain.invoke(prompt)

    return state

In [30]:
def create_blog(state: BlogState) -> BlogState:

    # Fetch title and outline
    title = state['title']
    outline = state['outline']

    # form a prompt
    prompt = f"Write a detailed blog on the title - {title} using the following outline \n {outline}"

    chain = llm | StrOutputParser()
    state['content'] = chain.invoke(prompt)

    return state

In [31]:
def evaluate_blog(state: BlogState) -> BlogState:

    # Fetch title and outline and content
    title = state['title']
    outline = state['outline']
    content = state['content']

    # Form a prompt
    prompt = f"Evaluate the blog content {content} on title - {title} which uses the following outline \n {outline}"

    chain = llm | StrOutputParser()
    state['evaluate'] = chain.invoke(prompt)

    return state

In [32]:
graph = StateGraph(BlogState)

# Add Nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# Add Edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

# Compile
workflow = graph.compile()

In [33]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)
print(final_state)
print(final_state['evaluate'])

{'title': 'Rise of AI in India', 'outline': 'Here is a detailed, SEO-friendly outline for a comprehensive blog post on the **"Rise of AI in India."** \n\n---\n\n# Blog Title Options:\n*   *Namaste AI: How India is Becoming a Global Artificial Intelligence Powerhouse*\n*   *The AI Revolution in India: Growth, Challenges, and the Future*\n*   *From IT Backoffice to AI Innovation Hub: The Rise of Artificial Intelligence in India*\n\n---\n\n## I. Introduction\n*   **The Hook:** Start with a compelling stat or visual (e.g., India’s massive digital transformation via Aadhaar, UPI, and smartphone penetration acting as the bedrock for AI).\n*   **Context:** Briefly explain the global AI boom (ChatGPT, generative AI) and how India has transitioned from being an IT services provider to an active creator of AI tech.\n*   **Thesis Statement:** India’s rise in the AI landscape is fueled by a massive talent pool, government backing, vibrant startup ecosystems, and unique socio-economic challenges th